# MCP Contract Validation — Walkthrough

This notebook demonstrates the **catalyst-llm-contract-mcp** validation pipeline:
1. Validate mention extractions against source text
2. See how invalid data gets rejected with typed error codes
3. Generate machine-readable repair instructions
4. Validate propositions with reference integrity checks

> **Kernel**: Run this from the `catalyst-langgraph-aio` venv:  
> `cd libs/catalyst-langgraph-aio && uv run jupyter notebook`

## Setup

In [ ]:
from catalyst_contracts.validators import (
    validate_mentions,
    validate_propositions,
    validate_spatial,
    generate_repair_plan,
)
from catalyst_contracts.models import ValidationResult, RepairPlan
import json

def pp(obj):
    """Pretty-print a pydantic model or dict."""
    if hasattr(obj, 'model_dump'):
        obj = obj.model_dump(mode='json')
    print(json.dumps(obj, indent=2))

## 1. Source Text

All mention validation checks spans against this source document.

In [ ]:
SOURCE_TEXT = (
    "The United Nations was founded in 1945 by 51 countries committed to "
    "maintaining international peace and security. Its headquarters is "
    "located in New York City, United States."
)

print(SOURCE_TEXT)
print(f"\nLength: {len(SOURCE_TEXT)} chars")

## 2. Valid Mentions

These mentions have correct spans that align with the source text.

In [ ]:
valid_mentions = [
    {
        "text": "United Nations",
        "mention_type": "ORG",
        "span_start": 4,
        "span_end": 18,
        "confidence": 0.95,
    },
    {
        "text": "New York City",
        "mention_type": "GPE",
        "span_start": 145,
        "span_end": 158,
        "confidence": 0.92,
    },
]

# Verify spans manually
for m in valid_mentions:
    actual = SOURCE_TEXT[m["span_start"]:m["span_end"]]
    print(f"  {m['text']!r} @ [{m['span_start']}:{m['span_end']}] => {actual!r} ✓" if actual == m["text"] else f"  MISMATCH: expected {m['text']!r}, got {actual!r}")

In [ ]:
result = validate_mentions(valid_mentions, SOURCE_TEXT, "doc-demo")

print(f"Verdict: {result.verdict.value}")
print(f"Valid: {result.valid_count}, Invalid: {result.invalid_count}")
print(f"Errors: {result.errors}")

## 3. Invalid Mentions — Span Mismatch

Now let's see what happens when the LLM gets the spans wrong.

In [ ]:
bad_mentions = [
    {
        "text": "United Nations",
        "mention_type": "ORG",
        "span_start": 0,   # Wrong! "The " is at 0, not "United"
        "span_end": 14,    # This gives "The United Nat" — wrong
        "confidence": 0.95,
    },
    {
        "text": "New York City",
        "mention_type": "BOGUS_TYPE",  # Invalid entity type
        "span_start": 145,
        "span_end": 158,
        "confidence": 1.5,  # Out of range [0, 1]
    },
]

result = validate_mentions(bad_mentions, SOURCE_TEXT, "doc-demo")

print(f"Verdict: {result.verdict.value}")
print(f"Valid: {result.valid_count}, Invalid: {result.invalid_count}")
print(f"\n{len(result.errors)} errors found:")
for e in result.errors:
    print(f"  [{e.code}] {e.message}")

## 4. Repair Instructions

The repair generator reads the validation errors and produces machine-readable fix instructions.
These get fed back to the LLM so it can self-correct.

In [ ]:
plan = generate_repair_plan(result, {"mentions": bad_mentions})

print(f"Repair plan: {len(plan.instructions)} instructions\n")
for inst in plan.instructions:
    print(f"  Action: {inst.action.value}")
    print(f"  Path:   {inst.path}")
    print(f"  Reason: {inst.reason}")
    if inst.suggested_value is not None:
        print(f"  Fix:    {inst.current_value} → {inst.suggested_value}")
    print(f"  Auto:   {inst.auto_applicable}")
    print()

## 5. Proposition Validation

Propositions are subject-predicate-object triples. The validator checks:
- Referenced mention IDs exist in the known set
- Predicates are snake_case
- Confidence is in [0, 1]

In [ ]:
# Valid proposition — no mention_id references, just text
valid_props = [
    {
        "kind": "binary",
        "subject_text": "United Nations",
        "predicate": "founded_in",
        "object_text": "1945",
        "confidence": 0.9,
    },
    {
        "kind": "binary",
        "subject_text": "United Nations",
        "predicate": "headquartered_in",
        "object_text": "New York City",
        "confidence": 0.88,
    },
]

known_ids = {"m-un", "m-nyc", "m-1945"}
result = validate_propositions(valid_props, known_ids, SOURCE_TEXT)

print(f"Verdict: {result.verdict.value}")
print(f"Valid: {result.valid_count}")

In [ ]:
# Invalid proposition — references a mention ID that doesn't exist
bad_props = [
    {
        "kind": "binary",
        "subject_text": "United Nations",
        "subject_mention_id": "m-nonexistent",  # Not in known_ids!
        "predicate": "founded_in",
        "object_text": "1945",
        "confidence": 0.9,
    },
]

result = validate_propositions(bad_props, known_ids, SOURCE_TEXT)

print(f"Verdict: {result.verdict.value}")
for e in result.errors:
    print(f"  [{e.code}] {e.message}")

## 6. Spatial Grounding Validation

For location mentions, the spatial validator checks coordinate bounds and geometry.

In [ ]:
spatial_candidates = [
    {
        "mention_text": "New York City",
        "lat": 40.7128,
        "lon": -74.0060,
        "confidence": 0.95,
    },
]

result = validate_spatial(spatial_candidates, SOURCE_TEXT)
print(f"Verdict: {result.verdict.value}  (Valid: {result.valid_count})")

# Now with impossible coordinates
bad_spatial = [
    {
        "mention_text": "Atlantis",
        "lat": 999.0,   # Impossible
        "lon": -74.0,
        "confidence": 0.5,
    },
]

result = validate_spatial(bad_spatial, SOURCE_TEXT)
print(f"Verdict: {result.verdict.value}")
for e in result.errors:
    print(f"  [{e.code}] {e.message}")

## 7. Full Validation Result Structure

Every validator returns the same `ValidationResult` shape — here's the full JSON.

In [ ]:
result = validate_mentions(bad_mentions, SOURCE_TEXT, "doc-demo")
pp(result)

---

**Next**: See [02_langgraph_extraction_pipeline.ipynb](./02_langgraph_extraction_pipeline.ipynb) for the full orchestration pipeline that uses these validators in an extract → validate → repair loop.